# Hamiltonian Expectation Values

This notebook demonstrates a practical quantum simulation task: estimating the expectation value of a Hamiltonian. This is a fundamental operation in quantum chemistry, materials science, and optimization.

## Key Concepts

1. **Hamiltonian decomposition**: Any Hamiltonian can be written as a sum of Pauli terms
2. **Term-wise estimation**: Measure each Pauli term separately
3. **Shot allocation**: Plan shots for each term based on its coefficient
4. **Error propagation**: Combine results with proper error analysis

In [1]:
import math
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2 as AerEstimator
from qiskit.transpiler import generate_preset_pass_manager

from qamp_shotplanner import (
    HoeffdingPlanner,
    hamiltonian_term,
    single_qubit_observable,
    correlation_observable,
)

## 1. The Transverse Ising Model

Let's work with a simple but important Hamiltonian: the Transverse Ising Model

$$H = -J\sum_{\langle i,j \rangle} Z_i Z_j - h\sum_i X_i$$

For 2 qubits with periodic boundary conditions:
$$H = -J(Z_0 Z_1 + Z_1 Z_0) - h(X_0 + X_1)$$

Let's use specific parameters: $J = 0.5, h = 0.3$
$$H = -0.25 \cdot Z_0 Z_1 - 0.3 \cdot X_0 - 0.3 \cdot X_1$$

In [2]:
# Hamiltonian parameters
J = 0.5  # Coupling strength
h = 0.3  # Transverse field

# Decompose H into Pauli terms
# H = term1 + term2 + term3
term1 = hamiltonian_term(
    qubits=(0, 1), paulis=("Z", "Z"), coefficient=-J/2, num_qubits=2
)  # -J/2 · ZZ (each pair counted twice)
term2 = hamiltonian_term(
    qubits=(0,), paulis=("X",), coefficient=-h, num_qubits=2
)  # -h · X_0
term3 = hamiltonian_term(
    qubits=(1,), paulis=("X",), coefficient=-h, num_qubits=2
)  # -h · X_1

print("Hamiltonian Decomposition:")
print(f"H = {term1.coeffs[0]:.4f} · {str(term1.paulis[0])}")
print(f"  + {term2.coeffs[0]:.4f} · {str(term2.paulis[0])}")
print(f"  + {term3.coeffs[0]:.4f} · {str(term3.paulis[0])}")

Hamiltonian Decomposition:
H = -0.2500+0.0000j · ZZ
  + -0.3000+0.0000j · IX
  + -0.3000+0.0000j · XI


## 2. Prepare Trial State

Let's prepare a simple trial state using a parameterized circuit:

In [3]:
# Prepare trial state using a simple ansatz
theta = math.pi / 4  # Variational parameter

qc_trial = QuantumCircuit(2)
qc_trial.ry(theta, 0)  # Rotate qubit 0
qc_trial.ry(theta/2, 1)  # Rotate qubit 1 differently
qc_trial.cx(0, 1)  # Add some entanglement (CNOT gate)

print(f"Trial State Circuit:")
print(f"Qubits: {qc_trial.num_qubits}")
print(f"Ansatz: Ry(θ={theta:.4f}) ⊗ Ry(θ/2={theta/2:.4f}) + CNOT")

Trial State Circuit:
Qubits: 2
Ansatz: Ry(θ=0.7854) ⊗ Ry(θ/2=0.3927) + CNOT


## 3. Shot Planning for Each Term

**Key insight**: Each term has a different coefficient, so we need different shot planning!

- **Term 1**: $c_1 = -0.25$, bounded in $[-0.25, +0.25]$
- **Term 2**: $c_2 = -0.3$, bounded in $[-0.3, +0.3]$
- **Term 3**: $c_3 = -0.3$, bounded in $[-0.3, +0.3]$

In [4]:
# Shot planning parameters
epsilon = 0.01  # Want total error ≤ 0.01
delta = 0.05    # Failure probability

# Plan shots for each term based on its coefficient range
coeffs = [term1.coeffs[0], term2.coeffs[0], term3.coeffs[0]]
terms = [term1, term2, term3]

print("Shot Planning for Hamiltonian Terms:")
print(f"Target: total error ≤ {epsilon} with probability ≥ {1-delta:.0%}\n")

shots_per_term = []
for i, (term, coeff) in enumerate(zip(terms, coeffs), 1):
    abs_coeff = abs(coeff)
    
    # Each term is bounded in [-|c|, +|c|]
    planner = HoeffdingPlanner(
        epsilon_stat=epsilon / 3,  # Allocate error budget across 3 terms
        delta=delta / 3,            # Allocate failure probability
        a=-abs_coeff,
        b=+abs_coeff,
    )
    shots_i = planner.planned_shots()
    shots_per_term.append(shots_i)
    
    print(f"Term {i}: {coeff:+.4f} · {str(term.paulis[0])}")
    print(f"  Range: [{-abs_coeff:.4f}, {+abs_coeff:.4f}]")
    print(f"  Planned shots: {shots_i:,}")
    print()

total_shots = sum(shots_per_term)
print(f"Total shots needed: {total_shots:,}")

Shot Planning for Hamiltonian Terms:
Target: total error ≤ 0.01 with probability ≥ 95%

Term 1: -0.2500+0.0000j · ZZ
  Range: [-0.2500, 0.2500]
  Planned shots: 53,860

Term 2: -0.3000+0.0000j · IX
  Range: [-0.3000, 0.3000]
  Planned shots: 77,558

Term 3: -0.3000+0.0000j · XI
  Range: [-0.3000, 0.3000]
  Planned shots: 77,558

Total shots needed: 208,976


## 4. Measure Each Hamiltonian Term

In [5]:
# Helper function to run estimator
def measure_observable(qc, observable, shots_val, seed=42):
    backend = AerSimulator()
    pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
    qc_isa = pm.run(qc)
    
    options = {
        "run_options": {"shots": shots_val}, 
        "backend_options": {"seed_simulator": seed}
    }
    estimator = AerEstimator(options=options)
    
    job = estimator.run([(qc_isa, observable, [])])
    return float(job.result()[0].data.evs)

# Measure each term
E_term1 = measure_observable(qc_trial, term1, shots_per_term[0])
E_term2 = measure_observable(qc_trial, term2, shots_per_term[1])
E_term3 = measure_observable(qc_trial, term3, shots_per_term[2])

print("Hamiltonian Term Expectations:")
print(f"⟨Term 1⟩ = {E_term1:.6f}  (ZZ correlation)")
print(f"⟨Term 2⟩ = {E_term2:.6f}  (X on qubit 0)")
print(f"⟨Term 3⟩ = {E_term3:.6f}  (X on qubit 1)")

Hamiltonian Term Expectations:
⟨Term 1⟩ = -0.230970  (ZZ correlation)
⟨Term 2⟩ = -0.081179  (X on qubit 0)
⟨Term 3⟩ = -0.114805  (X on qubit 1)


## 5. Combine Results: Total Energy

The total energy is the sum of all term expectations:

In [6]:
# Calculate total energy
total_energy = E_term1 + E_term2 + E_term3

print("Hamiltonian Expectation Value:")
print(f"⟨H⟩ = ⟨Term 1⟩ + ⟨Term 2⟩ + ⟨Term 3⟩")
print(f"⟨H⟩ = {E_term1:.6f} + {E_term2:.6f} + {E_term3:.6f}")
print(f"⟨H⟩ = {total_energy:.6f}")

print(f"\nInterpretation:")
print(f"- Energy of trial state: {total_energy:.6f}")
print(f"- Total shots used: {total_shots:,}")
print(f"- Error tolerance: ±{epsilon:.4f}")
print(f"- Confidence level: {1-delta:.0%}")

Hamiltonian Expectation Value:
⟨H⟩ = ⟨Term 1⟩ + ⟨Term 2⟩ + ⟨Term 3⟩
⟨H⟩ = -0.230970 + -0.081179 + -0.114805
⟨H⟩ = -0.426954

Interpretation:
- Energy of trial state: -0.426954
- Total shots used: 208,976
- Error tolerance: ±0.0100
- Confidence level: 95%


## 6. Error Analysis

Let's analyze how errors from each term contribute to the total error.

In [7]:
# Error propagation (worst case: add absolute errors)
term_errors = [epsilon / 3] * 3  # Each term gets 1/3 of error budget

print("Error Budget Allocation:")
for i, (error, shots) in enumerate(zip(term_errors, shots_per_term), 1):
    coeff = coeffs[i-1]
    print(f"Term {i}: error ≤ {error:.6f}, shots = {shots:,}")

total_error_bound = sum(term_errors)
print(f"\nTotal error bound: ≤ {total_error_bound:.6f}")
print(f"Target error: {epsilon:.6f}")
print(f"Satisfies guarantee: {total_error_bound <= epsilon}")

Error Budget Allocation:
Term 1: error ≤ 0.003333, shots = 53,860
Term 2: error ≤ 0.003333, shots = 77,558
Term 3: error ≤ 0.003333, shots = 77,558

Total error bound: ≤ 0.010000
Target error: 0.010000
Satisfies guarantee: True


## 7. Variational Optimization (Preview)

In practice, we'd vary the parameter θ to minimize the energy:

In [8]:
# Try different parameter values
thetas_to_test = [0, math.pi/8, math.pi/6, math.pi/4, math.pi/3]
energies = []

# Use fewer shots for exploration
exploration_shots = 5000

print("Energy vs Parameter θ:")
print("θ\tEnergy\tShots")
print("-" * 40)

for theta_test in thetas_to_test:
    # Prepare trial state with this theta
    qc_test = QuantumCircuit(2)
    qc_test.ry(theta_test, 0)
    qc_test.ry(theta_test/2, 1)
    qc_test.cx(0, 1)
    
    # Measure each term
    E1 = measure_observable(qc_test, term1, exploration_shots, seed=42)
    E2 = measure_observable(qc_test, term2, exploration_shots, seed=43)
    E3 = measure_observable(qc_test, term3, exploration_shots, seed=44)
    
    energy = E1 + E2 + E3
    energies.append(energy)
    
    print(f"{theta_test:.4f}\t{energy:.6f}\t{exploration_shots*3:,}")

min_energy = min(energies)
best_theta = thetas_to_test[energies.index(min_energy)]

print(f"\nBest energy found: {min_energy:.6f} at θ = {best_theta:.4f}")

Energy vs Parameter θ:
θ	Energy	Shots
----------------------------------------
0.0000	-0.250000	15,000
0.3927	-0.326121	15,000
0.5236	-0.357950	15,000
0.7854	-0.426954	15,000
1.0472	-0.496410	15,000

Best energy found: -0.496410 at θ = 1.0472


## 8. Key Takeaways

### Hamiltonian Estimation Workflow
1. **Decompose**: Write $H = \sum_i c_i \cdot P_i$ (Pauli decomposition)
2. **Plan shots**: Allocate shots based on each $|c_i|$ (coefficient magnitude)
3. **Measure**: Estimate each term separately using EstimatorV2
4. **Combine**: Sum term expectations: $\langle H \rangle = \sum_i c_i \langle P_i \rangle$

### Shot Planning Insights
- **Larger coefficients**: Need more shots (wider range $[-|c|, +|c|]$)
- **Smaller coefficients**: Need fewer shots (narrower range)
- **Error allocation**: Distribute error budget across terms

### Practical Applications
- **VQE (Variational Quantum Eigensolver)**: Find ground state energy
- **Quantum chemistry**: Estimate molecular energies
- **Materials science**: Study lattice Hamiltonians
- **Optimization**: Solve QUBO and Ising models

### Connection to All Previous Notebooks
- **Notebook 1**: SWAP test (single specific observable)
- **Notebook 2**: Single-qubit observables (X, Y, Z)
- **Notebook 3**: Multi-qubit correlations (ZZ, XX, YY)
- **Notebook 4**: Hamiltonians (sums of Pauli terms) ← **You are here**

The Hoeffding shot planner works for **all** of these scenarios!